# Lab 4: Smart Optimization Assistant using Dynamic Programming
## Design and Analysis of Algorithms (DAA)

### Overview
Dynamic Programming (DP) solves complex optimization problems by decomposing them into overlapping subproblems, solving each subproblem once, and tabulating results for instant reuse.

This lab implements a **Smart Optimization Assistant** encompassing:
1. **Sequence Analysis Module**: Longest Common Subsequence (LCS)
2. **Resource Allocation Module**: 0/1 Knapsack Problem
3. **Computation Optimization Module**: Matrix Chain Multiplication (MCM)
4. **Visualizations**: 2D DP Table Heatmaps & Traceback Highlights
5. **Performance & Complexity Analysis**: DP Tabulation vs Naive Recursion

## 1. Environment Setup & Module Imports

In [1]:
import sys
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure local modules can be loaded
sys.path.append(os.path.abspath('..'))

from algorithms.lcs import lcs, lcs_recursive, lcs_traceback_path
from algorithms.knapsack import knapsack_01, knapsack_space_optimized, knapsack_traceback_path, knapsack_recursive
from algorithms.matrix_chain import matrix_chain_order, print_optimal_parens, compute_standard_left_to_right_cost, matrix_chain_recursive
from analysis.visualizer import plot_dp_table, plot_dp_runtime_growth

print('All modules loaded successfully!')

---
## 2. Question 2: Sequence Analysis Module (Longest Common Subsequence)

Given two sequences $X$ and $Y$, finds the length and exact subsequence of characters appearing in both in order (not necessarily contiguously).

### State Transition Recurrence:
$$
dp[i][j] = \begin{cases}
0 & \text{if } i=0 \text{ or } j=0 \\
dp[i-1][j-1] + 1 & \text{if } X[i-1] == Y[j-1] \\
\max(dp[i-1][j], dp[i][j-1]) & \text{if } X[i-1] \neq Y[j-1]
\end{cases}
$$

**Time Complexity**: $O(m \times n)$  
**Space Complexity**: $O(m \times n)$ (reducible to $O(\min(m, n))$ for length)

In [2]:
x = "ABCBDAB"
y = "BDCABA"

length, subsequence, dp_table = lcs(x, y)
path = lcs_traceback_path(x, y)

print(f"Sequence X: {x}")
print(f"Sequence Y: {y}")
print(f"LCS Length: {length}")
print(f"LCS Subsequence String: {subsequence}")
print("\nDP Tabulation Table:")
dp_df = pd.DataFrame(dp_table, index=['Ø'] + list(x), columns=['Ø'] + list(y))
display(dp_df)

In [3]:
# Visualizing the LCS Table Heatmap with Traceback Path Highlight
plot_dp_table(
    dp_table,
    title=f"LCS DP Table: '{x}' vs '{y}'",
    xlabel=f"Sequence Y ({y})",
    ylabel=f"Sequence X ({x})",
    x_ticks=['Ø'] + list(y),
    y_ticks=['Ø'] + list(x),
    highlight_cells=path,
    cmap="Blues",
    save_filename="lcs_dp_table.png"
);

---
## 3. Question 3: Resource Allocation Module (0/1 Knapsack)

Given $n$ items each with weight $w_i$ and value $v_i$, choose a subset of items to maximize value without exceeding knapsack capacity $W$.

### State Transition Recurrence:
$$
dp[i][w] = \begin{cases}
dp[i-1][w] & \text{if } w_i > w \\
\max(dp[i-1][w], v_{i-1} + dp[i-1][w - w_{i-1}]) & \text{if } w_i \le w
\end{cases}
$$

**Time Complexity**: $O(n \times W)$ (pseudo-polynomial)  
**Space Complexity**: $O(n \times W)$ (reducible to $O(W)$ using 1D rolling array)

In [4]:
weights = [2, 3, 4, 5]
values = [3, 4, 5, 6]
capacity = 5

max_val, chosen, dp_table_kp = knapsack_01(weights, values, capacity)
opt_val_1d = knapsack_space_optimized(weights, values, capacity)
kp_path = knapsack_traceback_path(weights, values, capacity)

print(f"Weights: {weights}")
print(f"Values:  {values}")
print(f"Capacity: {capacity}")
print(f"Maximum Achievable Profit: {max_val}")
print(f"Selected Item Indices (0-indexed): {chosen}")
print(f"Selected Items Details:")
for idx in chosen:
    print(f"  - Item {idx+1}: Weight = {weights[idx]}, Value = {values[idx]}")
print(f"1D Rolling Array Space-Optimized Value: {opt_val_1d}")

y_labels = ['0 items'] + [f'Item {i+1} (w={weights[i]},v={values[i]})' for i in range(len(weights))]
x_labels = [f'w={w}' for w in range(capacity + 1)]
display(pd.DataFrame(dp_table_kp, index=y_labels, columns=x_labels))

In [5]:
# Visualizing the Knapsack DP Table with Optimal Selection Path
plot_dp_table(
    dp_table_kp,
    title=f"0-1 Knapsack DP Table (Capacity={capacity}, Max Value={max_val})",
    xlabel="Knapsack Capacity (w)",
    ylabel="Items Considered",
    x_ticks=x_labels,
    y_ticks=y_labels,
    highlight_cells=kp_path,
    cmap="Greens",
    save_filename="knapsack_dp_table.png"
);

---
## 4. Question 4: Computation Optimization Module (Matrix Chain Multiplication)

Given a chain of matrices $A_1, A_2, \dots, A_n$, finds the optimal parenthesization order that minimizes total scalar multiplications.

### State Transition Recurrence:
$$
cost[i][j] = \begin{cases}
0 & \text{if } i = j \\
\min_{i \le k < j} \{ cost[i][k] + cost[k+1][j] + p_{i-1} p_k p_j \} & \text{if } i < j
\end{cases}
$$

**Time Complexity**: $O(n^3)$  
**Space Complexity**: $O(n^2)$

In [6]:
dims = [10, 30, 5, 60]  # Matrices: A1(10x30), A2(30x5), A3(5x60)
min_cost, split_table, cost_table = matrix_chain_order(dims)
optimal_order = print_optimal_parens(split_table, 1, len(dims) - 1)
naive_cost = compute_standard_left_to_right_cost(dims)

print(f"Matrix Dimensions: {dims}")
print(f"Minimum Scalar Multiplications: {min_cost:,}")
print(f"Optimal Parenthesization Order: {optimal_order}")
print(f"Naive Left-to-Right Multiplication Cost: {naive_cost:,}")
print(f"Operations Saved: {naive_cost - min_cost:,} ({((naive_cost - min_cost)/naive_cost)*100:.1f}% reduction)")

In [7]:
n = len(dims) - 1
clean_cost = [[cost_table[i][j] if j >= i else 0 for j in range(1, n + 1)] for i in range(1, n + 1)]
ticks = [f"A{i}" for i in range(1, n + 1)]

plot_dp_table(
    clean_cost,
    title=f"MCM Cost Table (Optimal Cost={min_cost:,})",
    xlabel="Right Matrix Index (j)",
    ylabel="Left Matrix Index (i)",
    x_ticks=ticks,
    y_ticks=ticks,
    cmap="Purples",
    save_filename="mcm_cost_table.png"
);

---
## 5. Question 5: Complexity Comparison Table

In [8]:
comp_df = pd.read_csv('../data_csv/complexity_comparison.csv')
display(comp_df)

---
## 6. Question 6: Visualizations & Runtime Curves

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

lcs_df = pd.read_csv('../data_csv/lcs_benchmark.csv')
kp_df = pd.read_csv('../data_csv/knapsack_benchmark.csv')
mcm_df = pd.read_csv('../data_csv/matrix_chain_benchmark.csv')

axes[0].plot(lcs_df['String_Length_n'], lcs_df['Runtime_Seconds'], 'o-', color='#0066CC', lw=2)
axes[0].set_title('LCS Runtime Growth: O(m * n)', fontweight='bold')
axes[0].set_xlabel('Sequence Length (n)')
axes[0].set_ylabel('Execution Time (seconds)')
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].plot(kp_df['Num_Items_n'], kp_df['Runtime_2D_Seconds'], 's-', color='#107C41', lw=2, label='2D DP O(nW)')
axes[1].plot(kp_df['Num_Items_n'], kp_df['Runtime_1D_Seconds'], '^-', color='#F7941E', lw=2, label='1D DP O(W)')
axes[1].set_title('0/1 Knapsack Runtime Growth: O(n * W)', fontweight='bold')
axes[1].set_xlabel('Number of Items (n)')
axes[1].set_ylabel('Execution Time (seconds)')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

axes[2].plot(mcm_df['Num_Matrices_n'], mcm_df['Runtime_Seconds'], 'd-', color='#7B1FA2', lw=2)
axes[2].set_title('Matrix Chain Multiplication: O(n³)', fontweight='bold')
axes[2].set_xlabel('Number of Matrices (n)')
axes[2].set_ylabel('Execution Time (seconds)')
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

---
## 7. Question 7: Performance Analysis (DP Tabulation vs Naive Recursion)

In [10]:
rec_df = pd.read_csv('../data_csv/recursion_vs_dp.csv')
display(rec_df)

plt.figure(figsize=(8, 4.5))
plt.plot(rec_df['Sequence_Length'], rec_df['Naive_Recursive_Seconds'], 's-', color='#D9381E', label='Naive Recursion O(2ⁿ)')
plt.plot(rec_df['Sequence_Length'], rec_df['DP_Tabulation_Seconds'], 'o-', color='#107C41', label='DP Tabulation O(mn)')
plt.yscale('log')
plt.title('Logarithmic Comparison: Naive Recursion vs Dynamic Programming (LCS)', fontweight='bold')
plt.xlabel('Sequence Length (n)')
plt.ylabel('Runtime (seconds) [Log Scale]')
plt.legend()
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.show()